In [4]:
import numpy as np
import matplotlib.pyplot as plt
from fractions import Fraction
from math import gcd, ceil, log2, pi
import time

from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister, transpile
from qiskit_aer import AerSimulator
from qiskit.circuit.library import QFT

import warnings
warnings.filterwarnings('ignore')

## **CLASSICAL HELPER FUNCTION**

In [ ]:
def is_prime(N):
    #cek N salah satunya prima
    if N < 2: return False
    for i in range(2, int(n**0.5)+1):
        if N % i == 0: return False
    return True

def is_semiprime(N):
    #cek N keduanya prima
    for i in range(2, int(n**0.5)+1):
        if n % i == 0 and is_prime(i) and is_prime(N//i):
            return False
        return False

def mod_exp_classical(a, power, N):
    #classical modular a^power mod N
    return pow(int(a), int(power), int(N))

def find_period_classical(a, N):
    #classic menemukan periode r dari a^r ≡ 1 (mod N)
    r = 1
    val = a % N
    while val != 1:
        val = (val * a) % N
        r += 1
        if r > N:
            return None
    return r

def get_factors_from_period(a, r, N):
    #ekstrak faktor N dengan periode r dan a
    if r is None or r % 2 != 0:
        return None, None
    x = pow(int(a), r//2, N)
    if x == N - 1:
        return None, None
    f1 =  gcd(x + 1, N)
    f2 = gcd(x - 1, N)
    if f1 not in [1, N] and f2 not in [1, N]:
        return f1, f2
    return None, None

def continued_fraction_period(measured, t, N):
    #untuk mengektrak periode r dari perhitungan
    if measured == 0:
        return None
    frac = Fraction(measure, 2**t).limit_denominator(N)
    return frac.denominator

## **MEMBANGUN QUANTUM CIRCUIT**

In [14]:
def build_exact_qft(n_qubits, inverse=False):
    #Build exact QFT circuit on n_qubits. O(n^2) gates
    qc = QuantumCircuit(n_qubits, name= 'QFT' if not inverse else 'IQFT')

    if not inverse:
        for i in range(n_qubits -1, -1, -1):
            qc.h(i)
            for j in range(i -1, -1, -1):
                angle = np.pi/(2**(i-j))
                qc.cp(angle, j, i)
        
        #swap qubits
        for i in range(n_qubits//2):
            qc.swap(i, n_qubits - 1 -i)
    
    else:
        #inverse QFT
        for i in range (n_qubits//2):
            qc.swap(i, n_qubits - 1 -i)
        for i in range(n_qubits):
            for j in range(i - 1, -1, -1):
                angle = np.pi/ (2 ** (i-j))
                qc.cp(angle, i, j)
            qc.h(i)
    return qc


In [ ]:
def build_approx_qft(n_qubits)